In [ ]:
import pandas as pd
import numpy as np

train_woe = pd.read_csv("../data/processed/train_woe.csv")
validation_woe = pd.read_csv("../data/processed/validation_woe.csv")
test_woe = pd.read_csv("../data/processed/test_woe.csv")

final_variables = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "MonthlyIncome",
    "DebtRatio",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberRealEstateLoansOrLines",
    "NumberOfDependents",
    "NumberOfTimes90DaysLate"
]

In [ ]:
PDO = 20
BASE_SCORE = 600
BASE_ODDS = 20

In [ ]:
import numpy as np

PDO = 20

factor = PDO / np.log(2)

factor

In [ ]:
BASE_SCORE = 600
BASE_ODDS = 20

offset = (
    BASE_SCORE
    - factor * np.log(BASE_ODDS)
)

offset

In [ ]:
import statsmodels.api as sm

X = train_woe.drop(columns=["SeriousDlqin2yrs"])
y = train_woe["SeriousDlqin2yrs"]

X = sm.add_constant(X)

model = sm.Logit(y, X).fit()

model.summary()

In [ ]:
coefficients = model.params

coefficients

In [ ]:
# Calculate logit for validation set
X_validation = validation_woe[final_variables]

X_validation = sm.add_constant(X_validation)

validation_woe["logit"] = np.dot(
    X_validation,
    coefficients
)

In [ ]:
validation_woe[
    ["logit"]
].head()

In [ ]:
validation_woe["score"] = (
    offset
    - factor * validation_woe["logit"]
)

In [ ]:

X_validation = validation_woe[final_variables]

X_validation = sm.add_constant(X_validation)

validation_woe["pd"] = model.predict(
    X_validation
)

In [ ]:
validation_woe[
    ["pd", "score"]
].head()

In [ ]:
# Score distribution
validation_woe["score"].describe()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plt.hist(
    validation_woe["score"],
    bins=50
)

plt.title(
    "Score Distribution"
)

plt.xlabel("Score")
plt.ylabel("Count")

plt.show()

In [ ]:
#top 10 highest scores
validation_woe.sort_values(
    by="score",
    ascending=False
)[
    ["score", "pd"]
].head(10)

In [ ]:
#top 10 lowest scores
validation_woe.sort_values(
    by="score",
    ascending=True
)[
    ["score", "pd"]
].head(10)

## Scorecard Scaling

The logistic regression model was transformed into a credit score using a traditional scorecard scaling approach.

Parameters:

- Base Score = 600
- Base Odds = 20:1
- PDO = 20

The resulting scorecard produces higher scores for lower-risk customers and lower scores for higher-risk customers.

Each increase of 20 score points approximately doubles the odds of good versus bad performance.

In [ ]:
train_woe.to_csv(
    "../data/processed/train_scoring.csv",
    index=False
)

validation_woe.to_csv(
    "../data/processed/validation_scoring.csv",
    index=False
)

test_woe.to_csv(
    "../data/processed/test_scoring.csv",
    index=False
)